# Lấy mẫu dày video N-series + embedding PE-Core (Google Colab)

Video N (298 video camera giao thông cố định, ~10 phút/video) chỉ có 1–6 keyframe mỗi video vì cảnh gần như không đổi,
nên không thể tìm đúng thời điểm. Notebook này lấy **một khung mỗi `INTERVAL_SEC` giây (mặc định 2 giây, ≈ 88.000 khung)**
rồi embedding bằng đúng PE-Core đã dùng cho phần còn lại:

**tải video N từ `videos/` trên HF (nhanh, cùng trung tâm dữ liệu) → giải mã tuần tự, giữ khung round(k × INTERVAL_SEC × fps) → PE-Core → upload `.npz`**

**Trước khi chạy:** `Runtime → Change runtime type → A100 GPU` (nhiều nhân CPU càng tốt: giải mã video là phần chậm nhất).
Ước tính: khoảng 1 giờ trên A100 (GPU ≈ 30–35 ảnh/giây và giải mã chạy song song với GPU).

| Cell | Việc |
|---|---|
| 1 | Cài thư viện + kiểm tra GPU/CPU |
| 2 | Đăng nhập Hugging Face (token có quyền **Write**) |
| 3 | Cấu hình (`LIMIT = 3` để chạy thử 3 video trước) |
| 4 | Nạp PE-Core (~2–3 phút lần đầu) |
| 5 | Chạy |

**Kết quả** trên `5clnl/idk`: `embeddings_pecore_N_dense/<video>.npz` — `frame_idx` (int32), `pe_core` [n,1280] float32 chuẩn hoá, `fps`.
Có thể dừng bất cứ lúc nào: chạy lại cell 5 sẽ làm tiếp các video chưa có kết quả trên HF.
Ảnh xem trước (thumbnail) được cắt trên máy bạn từ video cục bộ, không phải upload/tải lại.


In [ ]:
#@title 1. Cài thư viện & kiểm tra GPU/CPU
# Ghim đúng phiên bản đang dùng ở máy local để kết quả khớp với phần đã làm.
!pip install -q "open_clip_torch==3.3.0" "timm==1.0.28" av

import os, torch
print("CPU:", os.cpu_count(), "nhân")
if not torch.cuda.is_available():
    print("CẢNH BÁO: không có GPU -> sẽ chạy rất chậm. Runtime > Change runtime type > chọn GPU.")
else:
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory/1e9:.0f} GB VRAM")


In [ ]:
#@title 2. Đăng nhập Hugging Face
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login, whoami
login()
print("Đã đăng nhập với tài khoản:", whoami()["name"])


In [ ]:
#@title 3. Cấu hình
REPO = "5clnl/idk"
VIDEO_DIR = "videos"                        # thư mục video trên HF
PREFIX = "N"                                # chỉ video tên bắt đầu bằng N (camera cố định)
INTERVAL_SEC = 2.0                          # một khung mỗi ... giây (phải giống lệnh cắt thumbnail trên máy: make_dense_thumbs.py --interval)
EMB_DIR = "embeddings_pecore_N_dense"       # kết quả embedding trên HF
NUM_DECODERS = 0                            # 0 = tự chọn theo số nhân CPU
UPLOAD_EVERY = 20                           # upload sau mỗi ... video (gộp commit cho HF khỏi giới hạn tần suất)
HEARTBEAT_SEC = 60                          # một dòng trạng thái mỗi ... giây khi chưa có video mới
LIMIT = 0                                   # 0 = tất cả video còn thiếu; đặt 3 để chạy thử
MIN_FRACTION = 0.85                         # video giải mã được ít hơn tỉ lệ này so với kế hoạch thì coi là lỗi
print(f"video '{PREFIX}*' trong {VIDEO_DIR}/, mỗi {INTERVAL_SEC:g}s một khung, kết quả -> {EMB_DIR}/")


In [ ]:
#@title 4. Nạp PE-Core + module lấy mẫu khung hình (cùng mã với máy local) + hàm embedding
#@markdown Embedding: PE-Core-bigG-14-448 fp16, chuẩn hoá L2 — giống hệt cách đã embed 614 video mới.
#@markdown Lấy mẫu: khung thứ round(k × INTERVAL_SEC × fps) của video, giải mã tuần tự (giống chỉ số frame đang có trong index).
import itertools, os, sys, time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np, torch, open_clip
from PIL import Image

# module lấy mẫu: ghi ra file để các tiến trình con (spawn) import được; cùng một mã với scripts/dense_sample.py trong repo
DENSE_SAMPLE_SRC = r'''"""Uniform (time-based) frame sampling of a video: one frame every `interval` seconds.

Used by BOTH sides of the N-series dense re-sampling so they pick exactly the same frames:
  * the Colab notebook (full-resolution JPEGs that are embedded with PE-Core), and
  * scripts/make_dense_thumbs.py (small thumbnails for the web UI).

Frame k is the decoded frame with index round(k * interval * fps); indices count decoded frames from 0, exactly like
the `frame_idx` of the keyframes already in the index (ms = frame_idx * 1000 / fps).  cv2 / av are imported lazily,
so a notebook can inline this file verbatim.
"""
import os


def plan(fps, n_frames, interval):
    """Frame indices to keep (strictly increasing) for a video of n_frames frames."""
    if n_frames <= 0:
        raise ValueError("unknown frame count")
    step = float(interval) * float(fps)
    out, k = [], 0
    while True:
        idx = int(round(k * step))
        if idx >= n_frames:
            break
        if not out or idx > out[-1]:
            out.append(idx)
        k += 1
    return out


def _probe_cv2(path):
    import cv2
    cap = cv2.VideoCapture(path)
    try:
        if not cap.isOpened():
            return 0, 0.0
        return int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    finally:
        cap.release()


def _probe_av(path):
    import av
    with av.open(path) as c:
        st = c.streams.video[0]
        fps = float(st.average_rate or 0)
        n = int(st.frames or 0)
        if n <= 0 and st.duration and fps:
            n = int(float(st.duration * st.time_base) * fps)
        return n, fps


def _decode_cv2(path, wanted):
    """Yield (idx, bgr) for the wanted indices (sorted), decoding sequentially; stops at the first unreadable frame."""
    import cv2
    cap = cv2.VideoCapture(path)
    try:
        i, w = 0, 0
        while w < len(wanted):
            if i == wanted[w]:
                ok, frame = cap.read()
                if not ok:
                    return
                yield i, frame
                w += 1
            elif not cap.grab():
                return
            i += 1
    finally:
        cap.release()


def _decode_av(path, wanted):
    import av
    want = set(wanted)
    last = wanted[-1] if wanted else -1
    with av.open(path) as c:
        s = c.streams.video[0]
        s.thread_type = "AUTO"
        for i, fr in enumerate(c.decode(s)):
            if i in want:
                yield i, fr.reformat(format="bgr24", src_colorspace="ITU601").to_ndarray()
            if i >= last:
                return


def sample_video(path, out_dir, interval, name=None, max_width=None, quality=95, decoder="auto"):
    """Write <name>_frame_<idx:06d>.jpg for every planned frame into out_dir.

    max_width: shrink (INTER_AREA) so the width is at most this; None keeps the full resolution.
    Returns {"video", "fps", "declared", "idx": [indices actually written], "planned", "decoder"}.
    """
    import cv2
    name = name or os.path.splitext(os.path.basename(path))[0]
    os.makedirs(out_dir, exist_ok=True)
    declared, fps = _probe_cv2(path)
    if declared <= 0 or fps < 1:                      # OpenCV cannot tell (or cannot open it): ask PyAV; it raises on garbage
        n_av, fps_av = _probe_av(path)
        declared, fps = declared if declared > 0 else n_av, fps if fps >= 1 else fps_av
    if declared <= 0:
        raise RuntimeError(f"{name}: cannot determine the number of frames")
    if fps < 1:
        fps = 25.0
    wanted = plan(fps, declared, interval)
    if not wanted:
        raise RuntimeError(f"{name}: nothing to sample (declared {declared} frames, fps {fps})")
    written, used = [], None
    for dec in (("cv2", "av") if decoder == "auto" else (decoder,)):
        gen = _decode_cv2(path, wanted) if dec == "cv2" else _decode_av(path, wanted)
        try:
            for idx, frame in gen:
                if max_width and frame.shape[1] > max_width:
                    h = max(1, round(frame.shape[0] * max_width / frame.shape[1]))
                    frame = cv2.resize(frame, (max_width, h), interpolation=cv2.INTER_AREA)
                ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
                if not ok:
                    raise RuntimeError(f"{name}@{idx}: JPEG encode failed")
                dst = os.path.join(out_dir, f"{name}_frame_{idx:06d}.jpg")
                tmp = dst + ".part"
                with open(tmp, "wb") as f:
                    f.write(buf.tobytes())
                os.replace(tmp, dst)
                written.append(idx)
        except Exception as e:                        # e.g. cv2 cannot decode this codec: try the next decoder
            if dec == "av" or decoder != "auto":
                raise
            written.clear()
            continue
        # accept when the decoder reached (almost) the end of the plan; otherwise let the next decoder try
        if len(written) >= 0.98 * len(wanted) or dec == "av" or decoder != "auto":
            used = dec
            break
        written.clear()
    if used is None or not written:
        raise RuntimeError(f"{name}: could not decode any planned frame")
    return {"video": name, "fps": fps, "declared": declared, "idx": written, "planned": len(wanted), "decoder": used}
'''
Path("/content").mkdir(exist_ok=True)
Path("/content/dense_sample.py").write_text(DENSE_SAMPLE_SRC, encoding="utf-8")
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import importlib
import dense_sample
importlib.reload(dense_sample)

EMB_DIM = 1280
EMBED_BATCH = 64
NUM_LOADERS = max(2, min(8, os.cpu_count() or 2))
LOAD_TIMEOUT = 300

device = "cuda" if torch.cuda.is_available() else "cpu"
pe_dtype = torch.float16 if device == "cuda" else torch.float32

print("Đang nạp PE-Core-bigG-14-448 (lần đầu tải ~5GB)...")
t0 = time.time()
pe_model, _, pe_pre = open_clip.create_model_and_transforms("hf-hub:timm/PE-Core-bigG-14-448", device="cpu")
if device == "cuda":
    pe_model = pe_model.half()
pe_model = pe_model.to(device).eval()
print(f"PE-Core sẵn sàng sau {time.time() - t0:.0f}s (CPU: {os.cpu_count()} nhân).")


def _load_batch(batch):
    return torch.stack([pe_pre(Image.open(p).convert("RGB")) for p in batch])


@torch.inference_mode()
def embed_paths(paths):
    """Embedding PE-Core (chuẩn hoá L2) cho danh sách ảnh, giữ nguyên thứ tự. Đọc ảnh bằng THREAD, có timeout."""
    out = np.empty((len(paths), EMB_DIM), np.float32)
    batches = [paths[i:i + EMBED_BATCH] for i in range(0, len(paths), EMBED_BATCH)]
    it = iter(batches)
    ex = ThreadPoolExecutor(max_workers=max(1, NUM_LOADERS))
    k = 0
    try:
        window = deque(ex.submit(_load_batch, b) for b in itertools.islice(it, max(2, NUM_LOADERS + 2)))
        while window:
            xb = window.popleft().result(timeout=LOAD_TIMEOUT)
            nxt = next(it, None)
            if nxt is not None:
                window.append(ex.submit(_load_batch, nxt))
            f = pe_model.encode_image(xb.to(device).to(pe_dtype))
            f = f / f.norm(dim=-1, keepdim=True).clamp_min(1e-8)
            assert f.shape[1] == EMB_DIM, f.shape
            out[k:k + len(f)] = f.float().cpu().numpy()
            k += len(f)
    finally:
        ex.shutdown(wait=False, cancel_futures=True)
    assert k == len(paths), (k, len(paths))
    return out


In [ ]:
#@title 5. Chạy: tải video → lấy mẫu mỗi INTERVAL_SEC giây → embedding → upload (có thể dừng/chạy lại, video xong rồi sẽ bỏ qua)
import multiprocessing as mp
import shutil, threading
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, ThreadPoolExecutor, wait
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

api = HfApi()
WORK = Path("/content/dense_work")
OUT = WORK / "out"
for d in (WORK / "dl", WORK / "frames", OUT):
    d.mkdir(parents=True, exist_ok=True)


def log(msg):
    print(msg, flush=True)


files = api.list_repo_files(REPO, repo_type="dataset")
src = {}
for f in files:
    p = Path(f)
    if f.startswith(VIDEO_DIR + "/") and p.suffix.lower() in (".mov", ".mp4") and p.stem.startswith(PREFIX):
        src[p.stem] = f
done_remote = {Path(f).stem for f in files if f.startswith(EMB_DIR + "/") and f.endswith(".npz")}
pending = sorted(v for v in src if v not in done_remote)
if LIMIT:
    pending = pending[:LIMIT]
log(f"{len(src)} video '{PREFIX}*' trên HF, đã xong {len(done_remote & set(src))}, sẽ làm {len(pending)} video "
    f"(mỗi {INTERVAL_SEC:g} giây một khung, ≈ {len(pending) * 600 / INTERVAL_SEC:,.0f} khung)")

n_dec = NUM_DECODERS or max(2, min(6, (os.cpu_count() or 4) // 2))
dec_pool = ProcessPoolExecutor(n_dec, mp_context=mp.get_context("spawn"))
prep_pool = ThreadPoolExecutor(n_dec + 1)
slots = threading.BoundedSemaphore(n_dec + 2)      # video đã giải mã nhưng chưa embedding: chặn đầy ổ đĩa
log(f"{n_dec} tiến trình giải mã song song")


def prepare(v):
    """Tải video v, lấy mẫu khung hình thành JPEG đủ độ phân giải. Trả về dict của dense_sample.sample_video."""
    slots.acquire()
    fdir = WORK / "frames" / v
    local = None
    try:
        last = None
        for attempt in range(1, 4):
            try:
                local = hf_hub_download(REPO, src[v], repo_type="dataset", local_dir=str(WORK / "dl"))
                break
            except Exception as e:
                last = e
                log(f"[{v}] tải lỗi lần {attempt}/3: {type(e).__name__}: {str(e)[:120]}")
                time.sleep(5 * attempt)
        else:
            raise RuntimeError(f"không tải được {src[v]}: {last}")
        shutil.rmtree(fdir, ignore_errors=True)
        return dec_pool.submit(dense_sample.sample_video, local, str(fdir), INTERVAL_SEC, v, None, 95).result()
    except BaseException:
        shutil.rmtree(fdir, ignore_errors=True)
        slots.release()
        raise
    finally:
        if local and os.path.exists(local):
            os.remove(local)


# file đã có sẵn trên HF (chạy lại cell này trong cùng phiên) thì không upload lại
uploaded = {p.name for p in OUT.glob("*.npz") if p.stem in done_remote}


def upload_outputs():
    """Đẩy các .npz mới lên HF (một commit cho cả nhóm) rồi kiểm tra kích thước từng file trên HF."""
    new = sorted(p for p in OUT.glob("*.npz") if not p.name.endswith(".tmp.npz") and p.name not in uploaded)
    if not new:
        return True
    for attempt in range(1, 5):
        try:
            api.upload_folder(repo_id=REPO, repo_type="dataset", folder_path=str(OUT), path_in_repo=EMB_DIR,
                              allow_patterns=[p.name for p in new], commit_message=f"PE-Core dense N ({len(new)} video, Colab)")
            info = {i.path: i.size for i in api.get_paths_info(REPO, [f"{EMB_DIR}/{p.name}" for p in new], repo_type="dataset")}
            bad = [p.name for p in new if info.get(f"{EMB_DIR}/{p.name}") != p.stat().st_size]
            if not bad:
                uploaded.update(p.name for p in new)
                log(f"   đã upload {len(new)} file lên {EMB_DIR}/ (đã kiểm tra kích thước)")
                return True
            log(f"   upload: {len(bad)} file sai kích thước (lần {attempt}/4): {bad[:3]}")
        except Exception as e:
            log(f"   upload lỗi lần {attempt}/4: {type(e).__name__}: {str(e)[:150]}")
        time.sleep(10 * attempt)
    return False


futs = {prep_pool.submit(prepare, v): v for v in pending}
waiting = set(futs)
done_ok, failed = [], []
n_frames, since_upload = 0, 0
t_start = t_last = time.time()
try:
    while waiting:
        ready, waiting = wait(waiting, timeout=HEARTBEAT_SEC, return_when=FIRST_COMPLETED)
        if not ready:
            log(f"[{time.strftime('%H:%M:%S')}] đang giải mã… xong {len(done_ok)}/{len(pending)} video, lỗi {len(failed)}"
                + (f"   ⚠ KẸT? {(time.time() - t_last) / 60:.0f} phút không có video mới" if time.time() - t_last > 600 else ""))
            continue
        for fu in ready:
            v = futs[fu]
            try:
                r = fu.result()
            except Exception as e:
                log(f"[{v}] !! LỖI khi tải/giải mã: {type(e).__name__}: {e}")
                failed.append(v)
                continue
            fdir = WORK / "frames" / v
            try:
                idx = r["idx"]
                if len(idx) < MIN_FRACTION * r["planned"]:
                    raise RuntimeError(f"chỉ giải mã được {len(idx)}/{r['planned']} khung (decoder {r['decoder']})")
                if len(idx) < 0.98 * r["planned"]:      # vài video kết thúc sớm hơn số khung khai báo trong file: bỏ phần đuôi không giải mã được
                    log(f"[{v}] lưu ý: {len(idx)}/{r['planned']} khung (phần đuôi video không giải mã được), vẫn dùng")
                t1 = time.time()
                embs = embed_paths([str(fdir / f"{v}_frame_{i:06d}.jpg") for i in idx])
                if not np.isfinite(embs).all() or abs(np.linalg.norm(embs, axis=1) - 1).max() > 1e-2:
                    raise RuntimeError("embedding không hợp lệ (NaN hoặc chưa chuẩn hoá)")
                tmp = OUT / f"{v}.tmp.npz"
                np.savez(tmp, frame_idx=np.array(idx, np.int32), pe_core=embs, fps=np.float32(r["fps"]))
                os.replace(tmp, OUT / f"{v}.npz")
                done_ok.append(v)
                n_frames += len(idx)
                since_upload += 1
                t_last = time.time()
                el = t_last - t_start
                left = el / len(done_ok) * (len(pending) - len(done_ok) - len(failed))
                log(f"[{len(done_ok) + len(failed)}/{len(pending)}] {v}: {len(idx)} khung ({r['decoder']}, {r['fps']:.2f} fps), embedding {t_last - t1:.0f}s"
                    f" | tổng {n_frames:,} khung, {el / 60:.1f} phút, còn ≈ {left / 60:.0f} phút")
            except Exception as e:
                log(f"[{v}] !! LỖI: {type(e).__name__}: {e}")
                failed.append(v)
            finally:
                shutil.rmtree(fdir, ignore_errors=True)
                slots.release()
            if since_upload >= UPLOAD_EVERY:
                if upload_outputs():
                    since_upload = 0
finally:
    dec_pool.shutdown(wait=False, cancel_futures=True)
    prep_pool.shutdown(wait=False, cancel_futures=True)

ok_up = upload_outputs()
log("\n================ TỔNG KẾT ================")
log(f"  xong {len(done_ok)}/{len(pending)} video, {n_frames:,} vector, {(time.time() - t_start) / 60:.1f} phút")
if failed:
    log("  Lỗi (chạy lại cell này để thử lại): " + ", ".join(sorted(failed)))
if not ok_up:
    log("  !! Còn file chưa upload được — chạy lại cell này (file vẫn nằm trong " + str(OUT) + ")")
if not failed and ok_up:
    log("Tất cả đã xong. Báo lại để tôi tải về, nạp vào chỉ mục.")
